In [1]:
from urllib.request import urlopen
import json
import pandas as pd

In [2]:
response = urlopen(
    'https://api.openf1.org/v1/race_control?flag=YELLOW&date%3E2023-01-01'
)

data = json.loads(response.read().decode('utf-8'))
yellow_flags = pd.DataFrame(data)

In [3]:
yellow_flags.head()

,meeting_key,session_key,date,driver_number,lap_number,category,flag,scope,sector,qualifying_phase,message
0,1140,7763,2023-02-24T11:00:06+00:00,None,NaN,Flag,YELLOW,Sector,5,NaN,YELLOW IN TRACK SECTOR 5
1,1140,7763,2023-02-24T16:15:14+00:00,None,NaN,Flag,YELLOW,Sector,11,NaN,YELLOW IN TRACK SECTOR 11
2,1140,9222,2023-02-23T11:00:47+00:00,None,NaN,Flag,YELLOW,Sector,10,NaN,YELLOW IN TRACK SECTOR 10
3,1140,9222,2023-02-23T11:24:51+00:00,None,NaN,Flag,YELLOW,Sector,15,NaN,YELLOW IN TRACK SECTOR 15
4,1140,9222,2023-02-23T11:30:12+00:00,None,NaN,Flag,YELLOW,Sector,7,NaN,YELLOW IN TRACK SECTOR 7


In [4]:
response = urlopen(
    'https://api.openf1.org/v1/weather?'
)

data = json.loads(response.read().decode('utf-8'))
weather = pd.DataFrame(data)

In [5]:
response = urlopen(
    'https://api.openf1.org/v1/sessions?'
)

data = json.loads(response.read().decode('utf-8'))
sessions = pd.DataFrame(data)

In [6]:
yellow_flags.to_csv('yellow_flags.csv', index=False)
weather.to_csv('weather.csv', index=False)
sessions.to_csv('sessions.csv', index=False)

In [9]:
# count how many session_types there are in the sessions dataframe
# and count how of the unique session_types there are in the sessions dataframe and their names

session_types_count = sessions['session_type'].nunique()
session_types_count

session_types_unique = sessions['session_type'].value_counts()
session_types_unique

session_type
Practice      311
Qualifying    154
Race          154
Name: count, dtype: int64

In [11]:
# create new data frame from yellow flag data where the session id is summed up and the number of yellow flags is counted for each session id

yellow_flag_summary = yellow_flags.groupby('session_key').size().reset_index(name='yellow_flag_count')
yellow_flag_summary.head()

,session_key,yellow_flag_count
0,7763,2
1,7765,2
2,7766,1
3,7772,1
4,7773,1


In [15]:
# take the averages of each weather variable for each session id and create a new data frame with the session id and the averages of each weather variable except date
#humidity,pressure,rainfall,track_temperature,wind_speed,meeting_key,wind_direction,air_temperature
weather_summary = weather.groupby('session_key').agg({
    'rainfall': 'mean',
    'air_temperature': 'mean',
    'humidity': 'mean',
    'wind_speed': 'mean',
    'pressure': 'mean',
    'track_temperature': 'mean',
    'wind_direction': 'mean'
}).reset_index()